# AI Agents Before LangGraph: A First-Principles Notebook

This notebook is designed to teach the **pre-LangGraph era of AI agents** from the ground up.

The goal is **not** to start with frameworks.  
The goal is to understand the **problems** people were trying to solve and how each new component was introduced to solve one specific limitation.

By the end of this notebook, you should understand:

- what problem a plain LLM/chatbot cannot solve by itself
- why people introduced loops, tools, memory, and planning
- how the **ReAct pattern** works
- how a simple agent can be built **manually**
- why frameworks like LangGraph appeared later

This notebook uses:
- plain Python
- small, readable examples
- mocked logic where needed, so the concepts stay clear
- no framework dependency

---

## Core promise of this notebook

At every stage, we will answer two questions:

1. **What problem do we currently have?**
2. **What new component are we adding, and exactly how does it solve that problem?**

That way, the idea and context stay consistent from beginning to end.

# Table of Contents

1. The core problem
2. A plain chatbot is not yet an agent
3. The minimal agent loop
4. Why actions/tools are needed
5. Building a tool abstraction
6. Why memory is needed
7. Adding working memory
8. Why planning/reasoning is needed
9. The ReAct pattern
10. Building a simple ReAct-style agent
11. Adding reflection and control
12. Multi-tool chaining and sequential reasoning
13. A better mental model of agent components
14. The pain points of building agents by hand
15. How LangGraph solves each pain point (with code)
16. Side-by-side comparison: manual vs LangGraph
17. One diagram to keep in your head
18. Final takeaways
19. Optional exercises
20. Closing note

---

## A note on realism

In production, people often use real LLM APIs.  
In this teaching notebook, we intentionally keep many parts **simple and explicit** so you can see the architecture clearly.

That is the right way to learn the **pre-framework mindset**.

# 1. Defining the Problem

Before discussing agents, we need to define the actual problem.

Suppose a user asks:

> "What is the weather in Chicago, and based on that, should I carry an umbrella?"

This looks simple, but it contains multiple hidden requirements:

1. The system needs to **understand the request**
2. It may need to **fetch external information**
3. It must **reason over that information**
4. It must produce a useful **actionable response**
5. In more complex tasks, it may need to **repeat this cycle** until it finishes

A plain one-shot response generator is often not enough.

---

## Important distinction

A normal chatbot often does this:

**Input -> Model -> Output**

An agent does something closer to this:

**Input -> Think -> Choose action -> Use tool -> Observe result -> Think again -> Answer**

That extra loop is the heart of the pre-LangGraph story.

# 2. A Plain Chatbot Is Not Yet an Agent

Let us start from the simplest system possible.

A plain chatbot takes user input and returns text.  
There is no explicit:
- tool use
- memory management
- repeated decision loop
- action selection
- observation from the outside world

It is just input/output.

That is useful, but limited.

In [ ]:
def plain_chatbot(user_input: str) -> str:
    """
    A deliberately tiny example of a plain chatbot.
    It only transforms input into a text response.
    There is no tool use, no memory, and no loop.
    """
    return f"I received your message: {user_input}"


print(plain_chatbot("Hello"))
print(plain_chatbot("What is the weather in Chicago?"))

## Problem with the plain chatbot

The chatbot can only **talk about** the task.  
It cannot **do** the task.

For example, if the user asks for current weather:
- the system would need access to some weather source
- it would need to decide to call that source
- it would need to process the returned information

A plain chatbot has no explicit machinery for this.

So the first problem is:

> **Problem 1: A one-shot chatbot cannot interact with the world in a structured way.**

We need something more than input/output text generation.

# 3. The Minimal Agent Loop

To move from chatbot to agent, the first conceptual upgrade is the **loop**.

An agent is not just something that speaks.  
An agent is something that repeatedly:

1. **observes**
2. **decides**
3. **acts**

Sometimes this is written as:

**Observe -> Think -> Act**

Sometimes learning is added later:

**Observe -> Think -> Act -> Learn**

The loop is the first key idea from the pre-framework era.

In [ ]:
def minimal_agent_once(observation: str) -> str:
    """
    The smallest possible "decision step".
    Here, our "thinking" is trivial and hard-coded.
    """
    if "hello" in observation.lower():
        return "Action: greet the user"
    return "Action: answer the user"


observation = "Hello there"
decision = minimal_agent_once(observation)

print("Observation:", observation)
print("Decision:", decision)

This still is not very powerful, but now we have shifted the mindset.

We are no longer thinking:

**input -> output**

We are thinking:

**observation -> internal decision -> action**

That difference is extremely important.

---

## What problem does the loop solve?

It solves this conceptual problem:

> **Problem 2: A chatbot has no explicit decision cycle.**

By introducing a loop, we now have a place to put:
- reasoning
- tool selection
- retries
- memory updates
- stopping conditions

Without a loop, all of those ideas remain hidden or impossible to control.

In [ ]:
def minimal_agent_loop(observations):
    """
    A tiny loop over observations, purely to show the structure.
    In a real agent, each action could change the next observation.
    """
    history = []
    for obs in observations:
        if "question" in obs.lower():
            action = "answer"
        else:
            action = "acknowledge"
        history.append((obs, action))
    return history


demo_history = minimal_agent_loop([
    "Hello",
    "I have a question about weather",
    "Thanks"
])

for i, (obs, action) in enumerate(demo_history, start=1):
    print(f"Step {i}: observation={obs!r}, chosen_action={action!r}")

# 4. Why Actions / Tools Are Needed

Now we hit the next problem.

Even with a loop, the agent still cannot do much unless it can affect or inspect the outside world.

For example:
- search for information
- calculate something
- look up a database value
- call an API
- read a file
- send a message

In the pre-LangGraph era, people often called these capabilities **tools**.

---

## The next problem

> **Problem 3: The agent can think, but it cannot actually do external work.**

So we add tools.

In [ ]:
def calculator_tool(expression: str) -> str:
    """
    Very small tool example.
    Only safe for demo expressions in this notebook.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Tool error: {e}"


print(calculator_tool("2 + 3 * 4"))
print(calculator_tool("(10 - 2) / 4"))

## What problem does a tool solve?

A tool solves the gap between:
- **thinking about the world**
- and **interacting with the world**

Without tools, the agent is trapped inside its own text generation.

With tools, the agent can:
- fetch
- compute
- transform
- inspect

That is why tools became central in early agent systems.

# 5. Building a Tool Abstraction

Early agent builders quickly discovered a new problem.

If you call tools ad hoc everywhere, your code becomes messy.

You need a consistent way to represent:
- the tool's name
- what the tool is for
- how to call it

So people began creating simple **tool abstractions**.

This is one of the earliest important engineering moves in the pre-framework era.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Dict


@dataclass
class Tool:
    name: str
    description: str
    func: Callable[[str], str]


def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Calculation failed: {e}"


def fake_weather(city: str) -> str:
    weather_db = {
        "chicago": "Light rain, 52 F",
        "houston": "Sunny, 78 F",
        "seattle": "Cloudy, 56 F",
    }
    return weather_db.get(city.lower().strip(), "Weather data not found")


TOOLS: Dict[str, Tool] = {
    "calculator": Tool(
        name="calculator",
        description="Use this to evaluate basic arithmetic expressions.",
        func=calculator
    ),
    "weather": Tool(
        name="weather",
        description="Use this to get a simple weather report for a city.",
        func=fake_weather
    )
}

for tool_name, tool in TOOLS.items():
    print(f"{tool_name}: {tool.description}")

In [ ]:
print("Calculator result:", TOOLS["calculator"].func("12 * 8 + 1"))
print("Weather result:", TOOLS["weather"].func("Chicago"))

## What problem does a tool abstraction solve?

It solves:

> **Problem 4: As capabilities grow, the code becomes unstructured and hard to manage.**

A tool abstraction gives us:
- a consistent interface
- a central registry
- easier routing
- easier debugging
- the ability to tell the model what tools exist

This is a major conceptual step toward later agent frameworks.

# 6. Why Memory Is Needed

Now consider this conversation:

User:
> "My city is Chicago."

Later:
> "Do I need an umbrella today?"

Without memory, the system may not know what "today" refers to or which city to use.

A stateless agent treats each step as isolated.  
But many tasks require continuity.

So we hit the next problem:

> **Problem 5: The agent forgets previous useful context.**

That is why memory enters the picture.

In [ ]:
conversation = [
    "My city is Chicago.",
    "Do I need an umbrella today?"
]

print("Without memory, each message would be processed in isolation:")
for msg in conversation:
    print("-", msg)

## Types of memory in early agent thinking

A simple and useful distinction is:

### 1. Working memory
Short-lived context needed for the current task.

Examples:
- recent conversation turns
- latest tool result
- current subgoal

### 2. Long-term memory
Persistent information across many interactions.

Examples:
- user preferences
- historical experiences
- retrieved knowledge

In early DIY agents, people often began with **working memory** first because it is simpler and immediately useful.

# 7. Adding Working Memory

We will now add a very simple memory object.

The goal is not sophistication.  
The goal is to solve the specific problem of lost context.

In [ ]:
class WorkingMemory:
    def __init__(self):
        self.items = []

    def add(self, item: str):
        self.items.append(item)

    def get_context(self) -> str:
        return "\n".join(self.items)


memory = WorkingMemory()
memory.add("User said their city is Chicago.")
memory.add("Latest task is about whether to carry an umbrella.")

print(memory.get_context())

Now the agent can consult memory before choosing what to do.

This is a crucial idea:

> The agent should not reason from the current input alone.  
> It should reason from **input + remembered context**.

That is a major leap in capability.

In [ ]:
def memory_aware_decision(user_input: str, memory: WorkingMemory) -> str:
    context = memory.get_context().lower()
    user_input_lower = user_input.lower()

    if "umbrella" in user_input_lower and "chicago" in context:
        return "Action: use weather tool for Chicago"
    return "Action: answer directly"


test_memory = WorkingMemory()
test_memory.add("User said their city is Chicago.")

print(memory_aware_decision("Do I need an umbrella today?", test_memory))
print(memory_aware_decision("Say hello", test_memory))

## What problem does memory solve?

It solves:

> **Problem 6: The agent cannot connect the current step to prior relevant information.**

Memory allows the agent to:
- preserve user context
- continue multi-step tasks
- carry forward tool outputs
- make future actions more informed

Without memory, the loop is blind to the past.

# 8. Why Planning / Reasoning Is Needed

Now suppose the user asks:

> "What is 15% tip on a $42 meal, and what is the final total?"

This requires more than raw memory and tools.  
The system may need to:
1. identify substeps
2. pick a tool
3. use the result
4. formulate the final answer

In more advanced tasks, reasoning becomes even more important:
- search, then summarize
- inspect data, then compare
- ask multiple tools in sequence
- revise after new observations

So we hit the next problem:

> **Problem 7: The agent needs an explicit process for deciding what to do next.**

That need leads to reasoning/planning patterns.

## A practical way to think about reasoning

In early LLM-agent work, reasoning was often represented in text like this:

- what do I know?
- what do I need?
- what should I do next?
- did the tool output solve the problem?
- do I stop or continue?

This did not always mean formal symbolic planning.  
Very often it meant **structured intermediate reasoning**.

In [ ]:
def simple_reasoning(user_request: str) -> str:
    request = user_request.lower()

    if "tip" in request and "$42" in request:
        return (
            "Reasoning:\n"
            "1. Need to compute 15 percent of 42.\n"
            "2. Then add that tip to 42.\n"
            "3. Use calculator tool."
        )
    return "Reasoning: answer directly."


print(simple_reasoning("What is 15% tip on a $42 meal, and what is the final total?"))

This is still simple, but now we have inserted an intermediate stage between input and action.

That intermediate stage is what later systems formalized more cleanly.

# 9. The ReAct Pattern

One of the most important pre-LangGraph ideas is the **ReAct** pattern:

**Reason + Act**

Instead of only generating a final answer, the model alternates between:
- **Thought**
- **Action**
- **Observation**

A typical trace looks like this:

- Thought: I need the weather in Chicago
- Action: weather("Chicago")
- Observation: Light rain, 52 F
- Thought: Rain is likely, so an umbrella is useful
- Answer: Yes, carry an umbrella

This pattern became extremely influential because it solved a very practical problem.

---

## What problem did ReAct solve?

> **Problem 8: One-shot reasoning is too limited for tasks that require external information and iterative decision-making.**

ReAct gave a structured way to combine:
- reasoning
- tool use
- feedback from the environment
- repeated decisions

In [ ]:
react_trace_example = [
    "Thought: The user asks about weather-based advice.",
    "Action: weather[Chicago]",
    "Observation: Light rain, 52 F",
    "Thought: Because rain is present, carrying an umbrella is recommended.",
    "Answer: Yes, carry an umbrella."
]

for line in react_trace_example:
    print(line)

## Why ReAct matters so much

ReAct is important because it makes the hidden agent loop visible.

Instead of vague internal processing, you now have a structure:

1. think
2. act
3. observe
4. think again
5. stop when ready

This is one of the clearest bridges from first-principles agents to later frameworks.

# 10. Building a Simple ReAct-Style Agent

Now we will build a small manual agent.

We will keep it intentionally simple and readable.

Our agent will have:
- a user request
- memory
- available tools
- a decision loop
- a stop condition

This is very close to how many people reasoned about agents before sophisticated frameworks.

In [ ]:
class SimpleReActAgent:
    def __init__(self, tools: Dict[str, Tool]):
        self.tools = tools
        self.memory = WorkingMemory()

    def think(self, user_input: str, latest_observation: str | None = None) -> str:
        """
        A very small rule-based 'thinking' function.
        In a real system, this would often be an LLM call.
        We keep it explicit here so the logic is visible.
        """
        text = user_input.lower()
        context = self.memory.get_context().lower()

        if latest_observation:
            if "light rain" in latest_observation.lower():
                return "ANSWER: Yes, you should carry an umbrella because rain is expected."
            if "sunny" in latest_observation.lower():
                return "ANSWER: No umbrella is likely needed based on the weather."
            return f"ANSWER: Based on the observation, here is the result: {latest_observation}"

        if "umbrella" in text:
            city = None
            if "chicago" in text or "chicago" in context:
                city = "Chicago"
            elif "seattle" in text or "seattle" in context:
                city = "Seattle"
            elif "houston" in text or "houston" in context:
                city = "Houston"

            if city:
                return f"ACTION: weather:{city}"
            return "ANSWER: I need to know the city before I can check weather."

        if "tip" in text and "42" in text:
            return "ACTION: calculator:(42 * 0.15), (42 + 42 * 0.15)"

        return "ANSWER: I can answer directly, but no external action is needed."

    def act(self, action_text: str) -> str:
        """
        Parses an action request and executes the corresponding tool.
        Expected format: ACTION: tool_name:tool_input
        """
        payload = action_text.replace("ACTION:", "", 1).strip()
        tool_name, tool_input = payload.split(":", 1)
        tool_name = tool_name.strip()
        tool_input = tool_input.strip()

        if tool_name not in self.tools:
            return f"Unknown tool: {tool_name}"

        return self.tools[tool_name].func(tool_input)

    def run(self, user_input: str, max_steps: int = 5) -> str:
        self.memory.add(f"User input: {user_input}")
        latest_observation = None

        for step in range(1, max_steps + 1):
            thought = self.think(user_input, latest_observation)
            print(f"Step {step} -> {thought}")

            if thought.startswith("ANSWER:"):
                answer = thought.replace("ANSWER:", "", 1).strip()
                self.memory.add(f"Final answer: {answer}")
                return answer

            if thought.startswith("ACTION:"):
                observation = self.act(thought)
                print(f"Step {step} observation -> {observation}")
                self.memory.add(f"Tool observation: {observation}")
                latest_observation = observation
                continue

            return "The agent reached an unexpected state."

        return "Stopped because the maximum number of steps was reached."

In [ ]:
agent = SimpleReActAgent(TOOLS)

answer_1 = agent.run("My city is Chicago. Do I need an umbrella today?")
print("\nFinal answer:", answer_1)

In [ ]:
agent2 = SimpleReActAgent(TOOLS)
answer_2 = agent2.run("What is 15% tip on a $42 meal, and what is the final total?")
print("\nFinal answer:", answer_2)

## Let us analyze what just happened

Our manual agent did the following:

1. stored the input in memory
2. decided whether it needed a tool
3. called the selected tool
4. observed the tool output
5. reasoned again
6. produced a final answer

That is a real agent loop, even though the reasoning logic is simple.

---

## What problem does this solve?

It solves:

> **Problem 9: We need a controllable process that can alternate between decision-making and external actions.**

This is the essence of pre-framework agent engineering.

# 11. Adding Reflection and Control

As soon as people built agent loops, they ran into another problem:

- agents can loop forever
- agents can choose poor actions
- tool output can be malformed
- reasoning can drift
- the system needs stopping conditions and safeguards

So people added control structures such as:
- max step limits
- validation
- error handling
- retries
- reflection

Reflection means the agent looks at what happened and asks:
- did this step help?
- do I need to correct course?
- should I stop?

This is a very important engineering layer.

In [ ]:
class ReflectiveAgent(SimpleReActAgent):
    def reflect(self, latest_observation: str) -> str:
        """
        A tiny reflection function.
        In real systems, this could be another LLM call or policy.
        """
        if "not found" in latest_observation.lower():
            return "Reflection: the tool did not return useful data; ask for clarification."
        if "failed" in latest_observation.lower() or "error" in latest_observation.lower():
            return "Reflection: the tool call failed; either retry or explain the issue."
        return "Reflection: the observation seems usable; continue reasoning."

    def run(self, user_input: str, max_steps: int = 5) -> str:
        self.memory.add(f"User input: {user_input}")
        latest_observation = None

        for step in range(1, max_steps + 1):
            thought = self.think(user_input, latest_observation)
            print(f"Step {step} -> {thought}")

            if thought.startswith("ANSWER:"):
                answer = thought.replace("ANSWER:", "", 1).strip()
                self.memory.add(f"Final answer: {answer}")
                return answer

            if thought.startswith("ACTION:"):
                observation = self.act(thought)
                print(f"Step {step} observation -> {observation}")
                self.memory.add(f"Tool observation: {observation}")

                reflection = self.reflect(observation)
                print(f"Step {step} {reflection}")
                self.memory.add(reflection)

                latest_observation = observation
                continue

            return "The agent reached an unexpected state."

        return "Stopped because the maximum number of steps was reached."

In [ ]:
reflective_agent = ReflectiveAgent(TOOLS)
answer_3 = reflective_agent.run("Check weather for Atlantis and tell me if I need an umbrella.")
print("\nFinal answer:", answer_3)

## What problem does reflection/control solve?

It solves:

> **Problem 10: Once an agent can loop and act, it also needs discipline and guardrails.**

This includes:
- preventing runaway loops
- handling missing data
- catching broken tool outputs
- keeping the process interpretable

This is one reason orchestration frameworks eventually became valuable: managing these control paths by hand gets complicated.

# 12. Multi-Tool Chaining and Sequential Reasoning

Our agent so far can call one tool per task. But real-world tasks often need **multiple tools in sequence**, where the output of one tool feeds into the next.

Example:

> "What is the weather in Chicago in Celsius?"

This requires:
1. Call the weather tool → get "Light rain, 52 F"
2. Extract the temperature (52)
3. Call the calculator tool → convert to Celsius: (52 - 32) * 5/9

This is called **tool chaining** and it was one of the hardest problems to solve cleanly before frameworks.

---

## The next problem

> **Problem 11: Many tasks require multiple tools called in sequence, where each step depends on the previous result.**

Without explicit chaining support, agents either:
- try to do everything in one step (and fail)
- lose intermediate results between steps
- have no structured way to pass data from one tool to another

In [ ]:
class ChainingAgent(ReflectiveAgent):
    """
    Extends the reflective agent with the ability to chain multiple tools.
    The key addition: a scratchpad that carries intermediate results
    so later tool calls can use earlier outputs.
    """

    def __init__(self, tools: Dict[str, Tool]):
        super().__init__(tools)
        self.scratchpad: list[dict] = []

    def think(self, user_input: str, latest_observation: str | None = None) -> str:
        text = user_input.lower()
        context = self.memory.get_context().lower()

        # Step 2: If we already have a Fahrenheit temp, convert it
        if latest_observation and self.scratchpad:
            last_step = self.scratchpad[-1]
            if last_step.get("tool") == "weather" and "celsius" in text:
                # Extract temperature number from weather result
                import re
                match = re.search(r"(\d+)\s*F", latest_observation)
                if match:
                    temp_f = match.group(1)
                    return f"ACTION: calculator:({temp_f} - 32) * 5 / 9"

            if last_step.get("tool") == "calculator" and len(self.scratchpad) > 1:
                weather_step = self.scratchpad[0]
                weather_info = weather_step.get("result", "")
                celsius = latest_observation
                return f"ANSWER: {weather_info}. That is approximately {float(celsius):.1f}°C."

        # Step 1: Need weather first
        if "weather" in text or "celsius" in text or "umbrella" in text:
            city = None
            for c in ["chicago", "seattle", "houston"]:
                if c in text or c in context:
                    city = c.title()
                    break
            if city:
                return f"ACTION: weather:{city}"

        return super().think(user_input, latest_observation)

    def act(self, action_text: str) -> str:
        """Execute action and record it in the scratchpad."""
        result = super().act(action_text)

        # Parse tool name from action
        payload = action_text.replace("ACTION:", "", 1).strip()
        tool_name = payload.split(":")[0].strip()

        self.scratchpad.append({
            "tool": tool_name,
            "action": action_text,
            "result": result,
        })
        return result

In [ ]:
chaining_agent = ChainingAgent(TOOLS)
answer_4 = chaining_agent.run("What is the weather in Chicago in Celsius?")
print("\nFinal answer:", answer_4)
print("\nScratchpad contents:")
for i, step in enumerate(chaining_agent.scratchpad, 1):
    print(f"  Step {i}: tool={step['tool']}, result={step['result']}")

## What did chaining require us to add?

Notice the growing complexity:
- A **scratchpad** to remember intermediate tool results
- Logic to **detect which step** we are on
- Logic to **extract data** from a previous tool's output
- Logic to **route** that data into the next tool

This works, but the code is becoming fragile. Every new chain pattern needs new if/else branches.

> **This is exactly the kind of complexity that motivated frameworks like LangGraph.**

The manual approach works for simple cases, but it does not scale.

# 13. A Better Mental Model of Agent Components

At this point, we can summarize the core components of a pre-LangGraph agent.

## Component 1: Environment
The world outside the agent.
Examples:
- user messages
- APIs
- files
- databases
- sensors

## Component 2: Perception
How the agent receives information from the environment.
Examples:
- text input
- parsed tool output
- retrieved documents

## Component 3: Memory
How the agent preserves context across steps.
Examples:
- conversation history
- latest observation
- saved preferences

## Component 4: Reasoning / Planning
How the agent chooses the next step.
Examples:
- "Do I answer now?"
- "Do I need a tool?"
- "Which tool should I call?"
- "Do I continue or stop?"

## Component 5: Action
How the agent affects or queries the world.
Examples:
- search
- compute
- call API
- write file
- send message

## Component 6: Control / Reflection
How the agent avoids getting lost.
Examples:
- max steps
- validation
- retries
- self-checks

## Component 7: State / Scratchpad
How the agent tracks intermediate results and progress.
Examples:
- tool chain results
- current sub-goal index
- accumulated data

These components did not appear because people wanted fancy architecture diagrams.  
They appeared because each one solved a real limitation.

# 14. The Pain Points of Building Agents by Hand

Now that we have built a complete manual agent, let us honestly evaluate what went wrong.

Look back at our code. To handle just a few simple tasks, we needed:

### Pain Point 1: Exploding if/else logic
Every new task pattern required adding more branches to `think()`. This does not scale. A real agent might need hundreds of routing decisions.

### Pain Point 2: State management is fragile
We added `WorkingMemory`, then a `scratchpad`, then tracked step counts manually. Each new feature required inventing a new state container.

### Pain Point 3: No clear flow visualization
Reading the code, it is hard to see "what happens after what." The control flow is buried in if/else chains inside a loop.

### Pain Point 4: Tool chaining is ad-hoc
Passing data between tools required manual parsing, regex extraction, and custom routing logic. Every new chain required new glue code.

### Pain Point 5: Error recovery is manual
Our `reflect()` method is simple, but real error recovery (retries, fallbacks, alternative paths) requires complex nested logic.

### Pain Point 6: No built-in persistence or checkpointing
If the agent crashes mid-task, all progress is lost. There is no way to resume from a checkpoint.

### Pain Point 7: No support for parallel or branching flows
What if we need to call two tools simultaneously? Or take different paths based on a condition? Our sequential loop cannot express this naturally.

---

## The core insight

> **The manual agent works, but it conflates two concerns:**
> 1. **What** the agent should do (the logic)
> 2. **How** the execution flows (the orchestration)
>
> Frameworks like LangGraph separate these concerns by letting you define the flow as a **graph** and the logic as **node functions**.

# 15. How LangGraph Solves Each Pain Point (With Code)

Now we will rebuild the **exact same weather agent** using LangGraph.

The goal is not to teach LangGraph in depth (that is covered in the other notebooks).  
The goal is to show **how each pain point from section 14 is addressed**.

---

## What is LangGraph?

LangGraph models agent behavior as a **state machine** (a directed graph):

- **Nodes** are functions that do work (call tools, reason, transform data)
- **Edges** define the flow between nodes
- **State** is a shared, typed object that all nodes can read and write
- **Conditional edges** replace if/else routing with clean branching logic

Instead of writing a monolithic loop with embedded routing, you declare:
- "After the reasoning node, go to the tool node OR the answer node"
- "After the tool node, always go back to the reasoning node"

The framework handles the execution loop, state passing, and control flow.

## Step 1: Define the State

In our manual agent, state was scattered across `self.memory`, `self.scratchpad`, local variables, and function parameters.

In LangGraph, you define **one typed state object** that flows through the entire graph.

> **Pain Point 2 solved: State management is centralized and typed.**

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict):
    """
    One single state object for the entire agent.
    Compare this to our manual agent where state was scattered
    across WorkingMemory, scratchpad, and local variables.
    """
    user_input: str
    reasoning: str
    tool_name: str
    tool_input: str
    tool_result: str
    final_answer: str
    step_count: int

## Step 2: Define the Nodes

Each node is a simple function that takes the state, does some work, and returns updated state fields.

Notice how clean each function is — it only cares about its own job.

> **Pain Point 1 solved: No exploding if/else. Each node has a single responsibility.**

In [ ]:
import re


# --- Node 1: Reasoning ---
def reasoning_node(state: AgentState) -> dict:
    """
    Decides what to do next based on the current state.
    In a real system, this would be an LLM call.
    Here we use rule-based logic to keep things visible.
    """
    user_input = state["user_input"].lower()
    tool_result = state.get("tool_result", "")
    step = state.get("step_count", 0)

    # If we already have a tool result, reason about it
    if tool_result:
        if "light rain" in tool_result.lower() or "cloudy" in tool_result.lower():
            return {
                "reasoning": "Weather suggests rain/clouds — umbrella recommended.",
                "final_answer": f"Based on the weather ({tool_result}), yes, carry an umbrella.",
                "tool_name": "",
                "step_count": step + 1,
            }
        elif "sunny" in tool_result.lower():
            return {
                "reasoning": "Weather is sunny — no umbrella needed.",
                "final_answer": f"Based on the weather ({tool_result}), no umbrella needed.",
                "tool_name": "",
                "step_count": step + 1,
            }
        else:
            return {
                "reasoning": f"Tool returned: {tool_result}",
                "final_answer": f"Here is what I found: {tool_result}",
                "tool_name": "",
                "step_count": step + 1,
            }

    # First pass: decide which tool to call
    if "weather" in user_input or "umbrella" in user_input:
        city = "Chicago"  # default
        for c in ["chicago", "seattle", "houston"]:
            if c in user_input:
                city = c.title()
                break
        return {
            "reasoning": f"Need weather data for {city}.",
            "tool_name": "weather",
            "tool_input": city,
            "step_count": step + 1,
        }

    if "tip" in user_input:
        match = re.search(r"\$(\d+)", user_input)
        amount = match.group(1) if match else "0"
        return {
            "reasoning": f"Need to calculate tip on ${amount}.",
            "tool_name": "calculator",
            "tool_input": f"{amount} * 0.15",
            "step_count": step + 1,
        }

    return {
        "reasoning": "Can answer directly.",
        "final_answer": "I can help with weather and calculations!",
        "tool_name": "",
        "step_count": step + 1,
    }


# --- Node 2: Tool Execution ---
def tool_node(state: AgentState) -> dict:
    """
    Executes the selected tool.
    Notice: this node does not decide WHICH tool — it just runs it.
    The routing decision is handled by edges (see below).
    """
    tool_name = state["tool_name"]
    tool_input = state["tool_input"]

    if tool_name in TOOLS:
        result = TOOLS[tool_name].func(tool_input)
    else:
        result = f"Unknown tool: {tool_name}"

    print(f"  [Tool Node] Called {tool_name}({tool_input!r}) → {result}")
    return {"tool_result": result}


print("Nodes defined: reasoning_node, tool_node")

## Step 3: Define the Edges (Flow Control)

This is where LangGraph truly shines.

Instead of writing nested if/else inside a while loop, you **declare** the flow:

- After reasoning: if a tool is needed → go to tool node. Otherwise → end.
- After tool execution: always → go back to reasoning.

The framework handles the loop for you.

> **Pain Point 3 solved: The flow is visible and declarative, not buried in code.**  
> **Pain Point 4 solved: Tool chaining happens naturally through the graph cycle.**

In [ ]:
def should_use_tool(state: AgentState) -> Literal["use_tool", "finish"]:
    """
    Conditional edge: decides where to go after reasoning.
    This replaces the if/else routing inside our manual loop.
    """
    if state.get("tool_name"):
        return "use_tool"
    return "finish"


# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("reason", reasoning_node)
workflow.add_node("tool", tool_node)

# Add edges
workflow.add_edge(START, "reason")                          # Start → Reasoning
workflow.add_conditional_edges("reason", should_use_tool, { # Reasoning → Tool or End
    "use_tool": "tool",
    "finish": END,
})
workflow.add_edge("tool", "reason")                         # Tool → back to Reasoning

# Compile
agent_graph = workflow.compile()

print("Graph compiled successfully!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print("Flow: START → reason → [tool → reason]* → END")

## Step 4: Run the LangGraph Agent

Now let us run the same queries we used with our manual agent.

In [ ]:
# Test 1: Weather query (same as our manual agent)
print("=" * 60)
print("TEST 1: Weather + umbrella advice")
print("=" * 60)

result_1 = agent_graph.invoke({
    "user_input": "Do I need an umbrella in Chicago?",
    "reasoning": "",
    "tool_name": "",
    "tool_input": "",
    "tool_result": "",
    "final_answer": "",
    "step_count": 0,
})

print(f"\nReasoning: {result_1['reasoning']}")
print(f"Final answer: {result_1['final_answer']}")
print(f"Steps taken: {result_1['step_count']}")

In [ ]:
# Test 2: Calculator query
print("=" * 60)
print("TEST 2: Tip calculation")
print("=" * 60)

result_2 = agent_graph.invoke({
    "user_input": "What is 15% tip on a $42 meal?",
    "reasoning": "",
    "tool_name": "",
    "tool_input": "",
    "tool_result": "",
    "final_answer": "",
    "step_count": 0,
})

print(f"\nReasoning: {result_2['reasoning']}")
print(f"Final answer: {result_2['final_answer']}")
print(f"Steps taken: {result_2['step_count']}")

In [ ]:
# Test 3: Direct answer (no tool needed)
print("=" * 60)
print("TEST 3: No tool needed")
print("=" * 60)

result_3 = agent_graph.invoke({
    "user_input": "Hello, what can you do?",
    "reasoning": "",
    "tool_name": "",
    "tool_input": "",
    "tool_result": "",
    "final_answer": "",
    "step_count": 0,
})

print(f"\nReasoning: {result_3['reasoning']}")
print(f"Final answer: {result_3['final_answer']}")
print(f"Steps taken: {result_3['step_count']}")

# 16. Side-by-Side Comparison: Manual vs LangGraph

Let us compare the two approaches directly.

| Concern | Manual Agent | LangGraph Agent |
|---|---|---|
| **State** | Scattered across `memory`, `scratchpad`, local vars | One `AgentState` TypedDict |
| **Control flow** | `while` loop + nested if/else | Declarative graph edges |
| **Tool routing** | String parsing inside `think()` | Conditional edge function |
| **Tool chaining** | Custom scratchpad + step detection | Natural graph cycle (tool → reason → tool) |
| **Adding a new tool** | Edit `think()`, add more if/else | Add tool to registry, node handles it |
| **Adding a new flow** | Rewrite the loop | Add a node and edge |
| **Visualization** | Read the code carefully | `graph.get_graph().draw_mermaid()` |
| **Persistence** | Not supported | Built-in checkpointing |
| **Error recovery** | Manual try/except + reflection | Retry policies, fallback edges |
| **Parallel execution** | Not supported | Fan-out / fan-in patterns |
| **Human-in-the-loop** | Custom input() calls | Built-in interrupt/resume |

---

## The key insight

The manual agent and the LangGraph agent do the **same work**.

But LangGraph separates:
- **What** each step does → node functions
- **When** each step runs → graph edges
- **What** data flows between steps → typed state

This separation is what makes the system maintainable as complexity grows.

# 17. One Diagram to Keep in Your Head

Here is the simplest conceptual picture of the journey from chatbot to LangGraph agent:

```text
═══════════════════════════════════════════════════════════════
  THE EVOLUTION: From Chatbot to Graph-Based Agent
═══════════════════════════════════════════════════════════════

  STAGE 1: Plain Chatbot          STAGE 2: Agent Loop
  ┌──────────┐                    ┌──────────┐
  │  Input   │                    │ Observe  │◄──────┐
  │    ↓     │                    │    ↓     │       │
  │  Output  │                    │  Think   │       │
  └──────────┘                    │    ↓     │       │
                                  │   Act    │───────┘
                                  └──────────┘

  STAGE 3: With Tools             STAGE 4: With Memory
  ┌──────────┐                    ┌──────────────────┐
  │  Think   │◄──────┐           │  Think + Memory  │◄──────┐
  │    ↓     │       │           │       ↓          │       │
  │  Tool?  ─┤─yes──►│           │     Tool?  ──────┤─yes──►│
  │    │     │  Tool  │           │       │          │ Tool   │
  │   no     │  Call  │           │      no          │ Call   │
  │    ↓     │       │           │       ↓          │       │
  │ Answer   │───────┘           │    Answer        │───────┘
  └──────────┘                    └──────────────────┘

  STAGE 5: LangGraph
  ┌─────────────────────────────────────────┐
  │                                         │
  │   START ──► [Reason Node]               │
  │                  │                      │
  │           ┌──────┴──────┐               │
  │           │             │               │
  │      needs tool    has answer            │
  │           │             │               │
  │     [Tool Node]       END               │
  │           │                             │
  │           └──────► [Reason Node]        │
  │                                         │
  │   State flows through every node.       │
  │   Edges define the control flow.        │
  │   The graph handles the loop.           │
  └─────────────────────────────────────────┘
```

This is the main idea.

If you understand this progression deeply, then everything in LangGraph becomes intuitive.

# 18. Final Takeaways

## The pre-LangGraph era is best understood as a sequence of problems and solutions

| # | Problem | Solution | Component Added |
|---|---------|----------|-----------------|
| 1 | A chatbot only talks, it cannot act | Introduce an observe-think-act loop | **Agent Loop** |
| 2 | No explicit decision cycle | Add a repeating decision structure | **Loop** |
| 3 | Cannot interact with the outside world | Add callable functions | **Tools** |
| 4 | Tool management becomes messy | Create a consistent tool abstraction | **Tool Registry** |
| 5 | Forgets previous context | Store and retrieve past information | **Working Memory** |
| 6 | Cannot connect current step to prior info | Reason from input + memory together | **Memory-Aware Reasoning** |
| 7 | No explicit process for deciding next steps | Add structured intermediate reasoning | **Planning / Reasoning** |
| 8 | One-shot reasoning is too limited | Alternate between thinking and acting | **ReAct Pattern** |
| 9 | Need controllable think-act alternation | Build a proper agent loop with tools | **ReAct Agent** |
| 10 | Agent can loop forever or drift | Add validation, retries, step limits | **Reflection / Control** |
| 11 | Multi-tool tasks need data passing | Track intermediate results explicitly | **Scratchpad / Chaining** |

## And then LangGraph solves the meta-problem

> **Problem 12: Managing all of these components by hand does not scale.**

LangGraph solves this by providing:
- **Typed state** instead of scattered variables
- **Graph edges** instead of nested if/else
- **Conditional routing** instead of manual string parsing
- **Built-in cycles** instead of while loops
- **Checkpointing** instead of no persistence
- **Visualization** instead of reading code

---

## The most important sentence in this notebook

> **An AI agent is a loop with memory, reasoning, actions, and feedback.  
> LangGraph is the framework that lets you build that loop as a clean, maintainable graph.**

# 19. Optional Exercises

To deepen your understanding, try these extensions:

### Manual Agent Exercises
1. Add a new tool called `unit_converter` that converts miles to kilometers
2. Modify `SimpleReActAgent` so it can chain three tools in sequence
3. Add a rule that stops the loop if the same action is repeated twice
4. Store the latest successful city in memory automatically so follow-up questions work

### LangGraph Exercises
5. Add a `validation` node to the LangGraph agent that checks tool output before passing it to reasoning
6. Add a second conditional edge that retries the tool if the result contains "not found"
7. Replace the rule-based `reasoning_node` with a real LLM call using `langchain_openai.ChatOpenAI`
8. Add a `memory_node` that persists user preferences across invocations using LangGraph's checkpointing

### Challenge Exercise
9. Build the same agent **three ways** — manual, LangGraph with rules, and LangGraph with a real LLM — and compare the code complexity of each approach

# 20. Closing Note

## What you now understand

After completing this notebook, you should be able to:

- Explain **why** each agent component exists (not just what it does)
- Build a working agent **from scratch** without any framework
- Identify the **specific limitations** that motivated frameworks
- Read LangGraph code and understand **which problem each part solves**
- Make an informed decision about **when to use a framework vs. building manually**

## Where to go next

This notebook covered the **conceptual foundation**. The other notebooks in this repository cover:

- **`langgraph_agents.ipynb`** — Building real LLM-powered agents with LangGraph
- **`langgraph_multi_agent_supervisor.ipynb`** — Multi-agent systems with supervisor patterns
- **`langgraph_custom_state_machines.ipynb`** — Advanced state machine patterns
- **`langgraph_human_in_the_loop.ipynb`** — Human-in-the-loop workflows

## The right learning order

**Problem first → component second → framework last.**

That is the right way to understand AI agents deeply.  
And now you have the foundation to do exactly that.